# Robot session

Everything needed to bring the robot up, calibrate it, and hand coordinates to a
picking routine. Run the cells in order the first time; afterwards the
calibration sections can be re-run on their own.

The order in section 1 is not optional. `move_to_coordinates`, `move_relative`
and `get_position` all require both a run and a loaded pipette, so anything that
moves the robot fails until `create_run` and `load_pipette` have happened.

## 0. Setup

In [ ]:
import os
# Notebooks live one level below the repository root.
os.environ["MICROPICK_ROOT"] = os.path.abspath("..")

import time
import numpy as np
import cv2

from opentrons_api import ot2_api

from micropick import paths
from micropick.config import store
from micropick.config.schema import CameraSpec, PipetteOffset
from micropick.hardware import labware
from micropick.hardware.camera import CameraManager
from micropick.hardware.protocols import xyz, goto_xy, move_to
from micropick.core.calibration.pixel_map import PixelMap, compare_degrees
from micropick.workflows.calibrate_camera import calibrate_camera
from micropick.workflows.calibrate_pipette import TipDetector, calibrate_pipette_offset
from micropick.workflows.jog import JogController, Limits, jog_in_window

paths.ensure_layout()
print(paths.describe())
print("\nprofiles:", store.list_profiles())

In [ ]:
from micropick.hardware import labware
for d in labware.list_definitions().values():
    print(d)

### Profile

One profile per installation. Created once, then loaded on every later run.

In [ ]:
# PROFILE = "lab_main"

# profile = store.create_profile(PROFILE, camera_label="overview_cam",
#                                notes="OT-2, gantry camera", exist_ok=True)
# print(profile)
# print("positions:", sorted(profile.positions) or "none yet")

In [ ]:
PROFILE = "lab_main"
profile = store.load_profile(PROFILE)
print(profile)
print("positions:", sorted(profile.positions) or "none yet")

### Cameras, first run only

Names must match what the operating system reports. Focus and exposure are
re-applied every time a camera is opened, which is what keeps a pixel map valid
across restarts: the map is only correct for the focus it was fitted at.

In [ ]:
from micropick.hardware import devices
print(*devices.list_devices(), sep="\n")

In [ ]:
profile.cameras = {
    "overview_cam": CameraSpec(
        device_name="20MP U3 Camera",
        resolutions=[[1280,720],
                    [1920,1080],
                    [2048,1536],
                    [2592,1944],
                    [3840,2160],
                    [4000,3000],
                    [4608,3456],
                    [5120,3840]],
        default_resolution=[2592, 1944], fps=30, fourcc="MJPG",
        controls={"auto_exposure": "manual"},
        notes="on the gantry, manual focus ring"),

    "underview_cam": CameraSpec(
        device_name="Arducam B0478 (USB3 48MP)",
        resolutions=[[1280,720],
                    [1920,1080],
                    [2000,1500],
                    [3840,2160],
                    [4000,3000],
                    [8000,6000]],
        default_resolution=[4000, 3000], fps=30, fourcc="MJPG",
        controls={"autofocus": 0, "focus": 920, "auto_exposure": "manual"},
        notes="tip calibration module, motorised focus"),
}
profile.save_cameras()
print(*profile.cameras.values(), sep="\n")

## 1. Robot

In [ ]:
openapi = ot2_api.OpentronsAPI()
openapi.add_slot_offsets([5, 8, 9], (0, 0, 64.2))

In [ ]:
openapi.toggle_lights()

In [ ]:
# Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

In [ ]:
# Once after power on.
openapi.home_robot()

In [ ]:
openapi.create_run()
openapi.load_pipette()
print("run:", openapi.run_id, " pipette:", openapi.pipette_id)

In [ ]:
openapi.move_to_coordinates((100, 100, 150))

### Labware

Definitions live in `labware/` and are uploaded into the current run. This has
to happen again after every `create_run`. Names and namespaces come from the
files themselves.

In [ ]:
for d in labware.list_definitions().values():
    print(d)

In [ ]:
labware.ensure_definitions(openapi)

TIP_RACK = "vwr_96_tiprack_200ul_xl"
labware.load_labware(openapi, TIP_RACK, 10)

In [ ]:
openapi.pick_up_tip(openapi.labware_dct["10"], "A4")

In [ ]:
openapi.move_labware(openapi.labware_dct['11'], 'offDeck')

## 2. Cameras

In [ ]:
cams = CameraManager.from_profile(profile)
over_cam = cams.open("overview_cam")
print(over_cam)

In [ ]:
# The lower camera is only needed for tip calibration; open it there.
# cams.close("underview_cam")

In [ ]:
def preview(camera, window="preview", size=(1348, 1011)):
    """Живой просмотр. Esc или q закрывает, s сохраняет кадр в outputs/images."""
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window, *size)
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                continue
            vis = frame.copy()
            h, w = vis.shape[:2]
            cv2.drawMarker(vis, (w // 2, h // 2), (0, 0, 255), cv2.MARKER_CROSS, 60, 2)
            cv2.putText(vis, f"{w}x{h}  {camera.measure_fps(0.0) if False else ''}"
                             f"frames {camera.frame_count}", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
            cv2.imshow(window, vis)
            key = cv2.waitKey(20) & 0xFF
            if key in (27, ord("q")):
                break
            if key == ord("s"):
                path = paths.images_dir() / f"{time.strftime('%H%M%S')}.png"
                cv2.imwrite(str(path), frame)
                print("saved", path)
    finally:
        cv2.destroyWindow(window)

# preview(over_cam)

### Jogging

Arrows or WASD move x and y, `q` and `e` move z, `+` and `-` change the step,
space saves a position, `u` undoes the last step, Enter finishes. The window
must have focus, so a stray keystroke in the notebook cannot drive the robot.

Limits are soft. Outside them the robot can always move back toward the working
area, only further out is refused.

In [ ]:
LIMITS = Limits(x=(0, 380), y=(0, 350), z=(0.1, 150))

def jog(title="", camera=None, step=1.0):
    ctrl = JogController(openapi, limits=LIMITS, step=step)
    pos = jog_in_window(ctrl, camera or over_cam, title=title)
    print("stopped at", tuple(round(v, 2) for v in pos))
    return pos

In [ ]:
jog()

## 3. Camera calibration

Fits lens distortion and the camera-to-robot relationship together from one
sweep of a static ArUco marker. No chessboard and no undistortion stage.

Redo it after any change to focus, zoom, camera height, or the height of the
plane the objects sit on.

In [ ]:
openapi.toggle_lights()

In [ ]:
MARKER_SIDE_MM = 6.8

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
params = cv2.aruco.DetectorParameters()
params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
detector = cv2.aruco.ArucoDetector(aruco_dict, params)

Put the marker roughly in the centre of the frame and set Z to the height you
actually image the dish at. The sweep keeps whatever Z it starts from.

In [ ]:
jog("centre the marker, set the working Z, then Enter")

In [ ]:
pmap, report, sweep = calibrate_camera(
    openapi, over_cam, detector,
    marker_side_mm=MARKER_SIDE_MM, grid_n=7, degree=3,
    on_progress=lambda i, n: print(f"  {i}/{n}", end="\r"))

print("\n")
print(report)

`track side` should come out near the printed marker size. The fit never uses
it, so agreement is independent evidence the sweep was good.

Degrees 1 and 2 should give identical numbers: radial distortion is cubic in
image coordinates, so a quadratic reduces exactly to an affine fit. A difference
between them would mean something other than lens distortion is in the data.

In [ ]:
print(compare_degrees(sweep.track_px, sweep.gantry, sweep.image_size,
                      marker_side_mm=MARKER_SIDE_MM))

In [ ]:
profile.calibration.pixel_map = pmap.to_config()
profile.save_calibration()
sweep.save(str(paths.fixtures_dir() / f"sweep_{time.strftime('%Y%m%d_%H%M')}.npz"))
print("saved")

### Closed-loop check

Ask the map where the marker is, drive there, and see how far it lands from the
reference pixel. This is the only test that includes the robot.

Read the spread, not the absolute value. A consistent offset in one direction
with a small spread is the camera-to-tip constant and belongs to the pipette
offset; scatter is the map and the robot's repeatability.

In [ ]:
def marker_centre_now():
    frame = over_cam.read_after(time.monotonic())
    corners, ids, _ = detector.detectMarkers(frame)
    if ids is None or len(corners) == 0:
        return None
    return corners[0].reshape(4, 2).mean(axis=0)

origin = xyz(openapi)
errors = []
for dx, dy in [(0, 0), (10, 7), (-12, -8), (18, -11), (-20, 12)]:
    openapi.move_to_coordinates((origin[0] + dx, origin[1] + dy, origin[2]-1),
                                min_z_height=1, verbose=False)
    time.sleep(0.4)
    g = xyz(openapi)[:2]                       # read next to the frame
    q = marker_centre_now()
    if q is None or not pmap.covers(*q):
        print(f"({dx:+3.0f},{dy:+3.0f}) not usable")
        continue

    target = pmap.to_robot(q[0], q[1], g)
    openapi.move_to_coordinates((target[0], target[1], origin[2]-1),
                                min_z_height=1, verbose=False)
    time.sleep(0.4)
    q2 = marker_centre_now()
    if q2 is None:
        continue
    err_px = q2 - np.array(pmap.config.ref)
    scale = float(np.mean(pmap.mm_per_px(*q2)))
    errors.append(err_px * scale)
    print(f"({dx:+3.0f},{dy:+3.0f})  residual {err_px[0]:+7.1f}, {err_px[1]:+7.1f} px"
          f"  = {np.linalg.norm(err_px) * scale * 1000:6.0f} um")

if errors:
    e = np.array(errors)
    print(f"\nbias   {e.mean(0)[0]*1000:+.0f}, {e.mean(0)[1]*1000:+.0f} um"
          f"   (constant, belongs to the pipette offset)")
    print(f"spread {np.linalg.norm(e - e.mean(0), axis=1).max()*1000:.0f} um max"
          f"   (this is the map plus robot repeatability)")

## 4. Pipette offset calibration

Redo this whenever a tip is picked up: every tip seats differently.

The upper camera locates the crosshair disc, the robot drives there using the
current offset, and the lower camera measures how far the tip actually is. The
gantry is parked a few millimetres to one side first, otherwise the tip covers
the crosshair and neither can be measured.

On a new installation the offset must be filled in roughly by hand first,
measured with a ruler. The routine drives to where it thinks the target is
before looking, so an offset that is wrong by tens of millimetres puts the tip
outside the lower camera's view and the run cannot recover.

In [ ]:
from ultralytics import YOLO

tip_model = YOLO(str(paths.ml_models_dir() / profile.calibration.tip_target.model_file))
tip_detector = TipDetector(tip_model,
                           imgsz=profile.calibration.tip_target.imgsz,
                           conf=profile.calibration.tip_target.conf)
under_cam = cams.open("underview_cam")
print(under_cam)

In [ ]:
openapi.toggle_lights()

In [ ]:
preview(under_cam)

First run only: fill in a rough offset measured with a ruler, and teach the
position of the calibration module.

In [ ]:
if profile.calibration.pipette_offset is None:
    profile.calibration.pipette_offset = PipetteOffset(
        dx=16.0, dy=60.0, tip_type="vwr_200ul_xl", method="manual")
    profile.save_calibration(backup=False)
print(profile.calibration.pipette_offset)

In [ ]:
# Teach where the crosshair disc is, once. Skip if it is already stored.
if "tip_calib" not in profile.positions:
    jog("bring the crosshair disc under the camera, then Enter")
    profile.remember("tip_calib", xyz(openapi))
print("tip_calib:", profile.where("tip_calib"))

In [ ]:
target = profile.calibration.tip_target
openapi.move_to_coordinates(profile.where("tip_calib"),
                            min_z_height=target.module_height - 0.1, verbose=False)
time.sleep(0.5)

`manual_touch_up` runs after the automatic correction, with the lower camera
live. Nudge the tip onto the crosshair with a small step if the result is not
good enough, then press Enter. Whatever it moves is included, because the offset
is read from the final pose rather than from the commanded moves.

In [ ]:
def touch_up(robot, camera, view):
    ctrl = JogController(robot, limits=LIMITS, step=0.05)
    jog_in_window(ctrl, camera, window="tip",
                  title="nudge the tip onto the crosshair, then Enter")

current = profile.calibration.pipette_offset
result = calibrate_pipette_offset(
    openapi, over_cam, under_cam, tip_detector, pmap,
    target=target,
    current_offset=(current.dx, current.dy),
    frames=7,
    tip_type=current.tip_type,
    manual_touch_up=touch_up)          # pass None to skip the manual step

print()
print(result)

In [ ]:
profile.calibration.pipette_offset = result.offset
profile.save_calibration(backup=False)
print("saved:", profile.calibration.pipette_offset)

### Camera homography

Maps upper-camera pixels to the lower camera, for placing a ROI box on the
pickup video. It comes free from the pipette calibration (both cameras saw the
same disc), and also runs on its own with no pipette and no moves. **It is valid
only for the marker plane and only at the gantry pose the upper frame was taken
at** — cuboids sit lower on the dish, so it is a coarse viewing aid, never a
positioning tool. Optional: skip it if there is no lower camera.

In [ ]:
# The pipette calibration above already saw the disc in both cameras, so it
# produced a homography for free. Keep it, or use the standalone cells below.
if result.homography is not None:
    profile.calibration.homography = result.homography
    profile.save_calibration(backup=False)
    print(result.homography_report)
else:
    print("no usable homography from the pipette run; try the standalone cell")

In [ ]:
from micropick.workflows.calibrate_homography import calibrate_homography

# Standalone: the disc under both cameras, no pipette, no moves. Reads the
# gantry pose the upper frame was taken at and stores it with the matrix.
hres = calibrate_homography(openapi, over_cam, under_cam, tip_detector,
                            target=profile.calibration.tip_target)
print(hres.report)

In [ ]:
# Per-point reprojection error, in lower-camera pixels. A large max means a
# crosshair was mis-detected or the correspondence is wrong.
for i, e in enumerate(hres.report.per_point_px):
    print(f"  point {i}: {e:.2f} px")
print(f"mean {hres.report.reproj_mean_px:.2f} px, "
      f"max {hres.report.reproj_max_px:.2f} px")

In [ ]:
profile.calibration.homography = hres.homography
profile.save_calibration(backup=False)
print("saved homography, captured at gantry", hres.homography.gantry_xy)

## 5. Using the calibration

`pixel_to_robot` is the one function the picking code needs. The gantry pose has
to be read next to the frame the pixel came from: the pose is part of the
conversion, not a correction applied afterwards.

`mm_per_px` replaces the old global size ratio. Scale varies by several percent
across the frame, so a single number misreports objects near the edges.

In [ ]:
profile = store.load_profile(PROFILE)
profile.require_calibration()
pmap = PixelMap.from_config(profile.pixel_map)
off = profile.calibration.pipette_offset
tip_offset = np.array([off.dx, off.dy])

problems = profile.pixel_map.check_camera(over_cam.resolution)
if problems:
    raise RuntimeError("the calibration does not match the camera: " + "; ".join(problems))

def pixel_to_robot(u, v, gantry_xy):
    """Robot coordinates that put the pipette tip on the pixel (u, v)."""
    if not pmap.covers(u, v):
        raise ValueError(f"pixel ({u:.0f}, {v:.0f}) is outside the calibrated area")
    return pmap.to_robot(u, v, gantry_xy) + tip_offset

def area_mm2(area_px, u, v):
    su, sv = pmap.mm_per_px(u, v)
    return area_px * su * sv

print("ready:", pmap.config.degree, "degree map,",
      f"holdout {pmap.config.holdout_mean_um:.1f} um,",
      f"offset ({off.dx:.2f}, {off.dy:.2f}) mm")

In [ ]:
# Example: convert one detection.
g = xyz(openapi)[:2]
frame = over_cam.read_after(time.monotonic())
# u, v = ...detect something...
# tx, ty = pixel_to_robot(u, v, g)

In [ ]:
profile.where("tip_calib")

In [ ]:
def click_to_go(z=None, snap_px=60, conf=0.25, imgsz=2016, move=True):
    """
    Клик по кресту -> пипетка едет туда.

    Клик привязывается к ближайшей детекции, а не к сырым координатам курсора:
    попасть мышью в пиксель невозможно, а центр бокса модели субпиксельный.

    После переезда тот же крест находится заново, и его координаты считаются
    из новой позы гантри. Совпадение с прежними это и есть проверка карты по
    полю, для неё не нужно ничего измерять руками.

    Клавиши: d пересчитать детекции, r сбросить статистику, Esc выход.
    """
    win = "click to go"
    state = {"dets": [], "click": None, "frame": None, "gantry": None}
    history = []

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            state["click"] = (x, y)

        if event == cv2.EVENT_RBUTTONDOWN:
            openapi.move_to_coordinates(profile.where("tip_calib"))

    def detect_now():
        g = np.array(xyz(openapi)[:2])
        frame = over_cam.read_after(time.monotonic())
        res = tip_model.predict(source=frame[..., ::-1], conf=conf, imgsz=imgsz,
                                save=False, verbose=False)
        pts = []
        for r in res:
            for b in r.boxes:
                if tip_model.names[int(b.cls[0])] != "point":
                    continue
                x1, y1, x2, y2 = (float(v) for v in b.xyxy[0])
                p = np.array([(x1 + x2) / 2, (y1 + y2) / 2])
                if pmap.covers(*p):
                    pts.append((p, pmap.to_robot(p[0], p[1], g)))
        state["dets"], state["gantry"] = pts, g
        return pts

    print("детекция...")
    detect_now()
    print(f"найдено {len(state['dets'])} крестов")

    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, 1348, 1011)
    cv2.setMouseCallback(win, on_mouse)
    try:
        while True:
            ok, frame = over_cam.read()
            if not ok:
                continue
            vis = frame.copy()
            h, w = vis.shape[:2]
            cv2.drawMarker(vis, tuple(np.int32(pmap.config.ref)), (0, 0, 255),
                           cv2.MARKER_CROSS, 60, 2)
            for p, world in state["dets"]:
                cv2.circle(vis, tuple(np.int32(p)), 14, (0, 255, 0), 2)
            cv2.putText(vis, f"{len(state['dets'])} crosses   click one   "
                             f"d=redetect  r=reset  Esc=quit", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
            if history:
                e = np.array(history)
                cv2.putText(vis, f"map consistency: mean {e.mean()*1000:.0f} um, "
                                 f"max {e.max()*1000:.0f} um  (n={len(e)})",
                            (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                            (0, 200, 255), 2)
            cv2.imshow(win, vis)

            key = cv2.waitKey(20) & 0xFF
            if key == 27:
                break
            if key == ord("d"):
                detect_now(); print(f"найдено {len(state['dets'])}")
            if key == ord("r"):
                history.clear()

            if state["click"] is None:
                continue
            cx, cy = state["click"]
            state["click"] = None
            if not state["dets"]:
                print("нет детекций, нажми d"); continue

            # привязка к ближайшему кресту в координатах отображаемого кадра
            # scale_x = frame.shape[1] / cv2.getWindowImageRect(win)[2]
            # scale_y = frame.shape[0] / cv2.getWindowImageRect(win)[3]
            # click_px = np.array([cx * scale_x, cy * scale_y])
            click_px = np.array([cx, cy])
            d = [np.linalg.norm(p - click_px) for p, _ in state["dets"]]
            k = int(np.argmin(d))
            # if d[k] > snap_px * max(scale_x, scale_y):
            #     print(f"мимо креста ({d[k]:.0f} px до ближайшего)"); continue
            if d[k] > snap_px:
                print(f"мимо креста ({d[k]:.0f} px до ближайшего)"); continue

            px, world = state["dets"][k]
            target = world + tip_offset
            print(f"\nкрест на пикселе ({px[0]:.0f}, {px[1]:.0f})")
            print(f"  координата креста : {world.round(3)}")
            print(f"  цель для пипетки  : {target.round(3)}")
            if not move:
                continue

            goto_xy(openapi, target[0], target[1])
            if z is not None:
                openapi.move_to_coordinates((target[0], target[1], z), min_z_height=1, verbose=False)
                # move_to(openapi, (target[0], target[1], z))

            # тот же крест заново, из новой позы
            after = detect_now()
            if after:
                worlds = np.array([wr for _, wr in after])
                j = int(np.argmin(np.linalg.norm(worlds - world, axis=1)))
                drift = float(np.linalg.norm(worlds[j] - world))
                history.append(drift)
                print(f"  тот же крест из новой позы: {worlds[j].round(3)}")
                print(f"  расхождение карты: {drift*1000:.0f} um")
    finally:
        cv2.destroyWindow(win)

    if history:
        e = np.array(history)
        print(f"\nсогласованность карты по {len(e)} переездам: "
              f"среднее {e.mean()*1000:.0f} мкм, максимум {e.max()*1000:.0f} мкм")
    return history



In [ ]:
jog("Alignment")

In [ ]:
drift = click_to_go(z = 67.0)

## 6. Picking cuboids

The picking session (`workflows/picking.py`) is headless: it has no loop, no
window and no keyboard. This section is the external layer that provides them.
`session.step()` performs one transition and returns an event; the run cell
holds the `while`, the pause, the stop and the display, and reads keys from the
focused window rather than global hooks, so a stray keypress in the notebook
cannot drive the robot.

Run the cells in order: detector, config, plate map, routine, (optional) logger,
then the run cell. Section 5 must have run first, so `pmap`, `over_cam` and
`profile` exist.

In [ ]:
# --- Detector -----------------------------------------------------------
# The cuboid YOLO weights live in ml_models/ and are not tracked in the repo.
import threading
from ultralytics import YOLO

from micropick.core import routine as rt
from micropick.workflows.picking import PickingSession, RobotState
from micropick.viz import overlays

CUBOID_WEIGHTS = "cuboid_bbox_v4-11_best.pt"      # change to your weights file
_weights = paths.ml_models_dir() / CUBOID_WEIGHTS
if not _weights.exists():
    raise FileNotFoundError(
        f"cuboid detector weights not found: {_weights}\n"
        f"put the .pt file in {paths.ml_models_dir()} "
        f"(weights are not tracked in the repository)")
cuboid_model = YOLO(str(_weights))
print("loaded", CUBOID_WEIGHTS)

In [ ]:
# --- Config -------------------------------------------------------------
# Picking parameters come from the profile. Edit in place for this run; the
# commented save writes the change back to picking.json.
cfg = profile.picking
cfg.miss_policy = "keep_successful"        # or "return_all"
cfg.vol = 10.0
cfg.max_batch = 10
# profile.save_picking()
print(cfg.miss_policy, "| vol", cfg.vol, "| batch", cfg.max_batch,
      "| slot", cfg.destination_slot)

In [ ]:
# --- Plate map ----------------------------------------------------------
# Load the destination plate into its slot, then say how many cuboids go into
# each well. SIZE must match the loaded plate. Fill the table by hand, the old
# way: well_df.loc['C', 3] = 1
SIZE = 96
PLATE = "wide_bore_200ul"                 # a load_name from labware/
if str(cfg.destination_slot) not in openapi.labware_dct:
    labware.load_labware(openapi, PLATE, cfg.destination_slot)
labware_id = openapi.labware_dct[str(cfg.destination_slot)]

well_df = rt.empty_plate_table(SIZE)
well_df.loc['C', 3] = 1
# well_df.loc['D', 5] = 2
# ...

plan = rt.plan_from_table(well_df, SIZE)
dest = rt.Destination.plate(SIZE, cfg.destination_slot)
PROGRESS = paths.outputs_dir() / f"picking_{PROFILE}.json"
print(len(plan), "wells,", sum(plan.values()), "cuboids ->", dest)
print("progress file:", PROGRESS)

### Routine — new run (erases saved progress)

A routine tracks how many cuboids have gone into each well and writes that to
disk after every pickup. Creating a new one starts that record from zero, so
the constructor is commented out on purpose — run it only when starting over,
not after a restart mid-plate.

In [ ]:
# DANGER: uncomment to start a NEW run. This overwrites PROGRESS on disk and
# discards the record of everything already filled. Leave it commented unless
# you really are starting the plate over.
# routine = rt.Routine(dest, plan, strategy="by_row", path=PROGRESS)
# routine.save()
# print("new routine:", routine)

### Routine — continue after a kernel restart

This is the cell to run after a restart: it loads the saved progress and picks
up exactly where the previous run stopped.

In [ ]:
routine = rt.Routine.load(PROGRESS)
left = sum(routine.remaining(t) for t in routine.plan)
print(routine, "| cuboids remaining:", left)

In [ ]:
# --- Logger (optional) --------------------------------------------------
logger = None      # run without logging

# To keep a run log, pass any object with a .log(str) method:
# import datetime
# class RunLog:
#     def __init__(self, path):
#         self.path = path
#     def log(self, msg):
#         with open(self.path, "a", encoding="utf-8") as f:
#             f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}  {msg}\n")
# logger = RunLog(paths.logs_dir() / f"picking_{PROFILE}.log")

### Run

`session.step()` runs in a worker thread while this cell's loop owns the window
and the keys. That split is deliberate: `pause` blocks *inside* `step()` between
moves, so a pause during a five-cuboid pickup takes effect at once. A single
thread that both stepped and read keys would deadlock the moment it paused, with
nothing left to read the un-pause key.

Keys (the window must have focus): **p** pause/resume, **Esc** stop the routine,
**q** leave the window without stopping it, **r** resume after `NEEDS_OPERATOR`.

In [ ]:
pause, stop = threading.Event(), threading.Event()

# Optional lower-camera clip per pickup. None = off: no recorder is created and
# no frames accumulate. To record, set a folder and make sure under_cam is open
# (section 4). A homography in the profile places the ROI box; without one the
# clip records with no box, which is not an error.
CLIP_DIR = None                          # e.g. paths.clips_dir() / PROFILE

session = PickingSession(openapi, over_cam, pmap, profile, routine,
                         cuboid_model, labware_id=labware_id, logger=logger,
                         under_cam=(under_cam if CLIP_DIR else None),
                         clip_dir=CLIP_DIR)

# The session writes its latest detections here whenever it emits a frame; the
# display reads them under the lock. The base video comes straight from the
# camera, so the window stays smooth between detections.
lock = threading.Lock()
snap = {"df": None, "pickable": None, "isolated": None, "zones": (),
        "state": session.state, "target": None}

def on_frame(frame, df):
    with lock:
        snap.update(df=df, pickable=session.pickable, isolated=session.isolated,
                    zones=session.floater_zones, state=session.state,
                    target=session.routine.current)
session.on_frame = on_frame

def worker():
    while not session.done:
        event = session.step(pause=pause, stop=stop)
        print(event)
        if session.state is RobotState.NEEDS_OPERATOR:
            time.sleep(0.2)                 # wait for the operator, do not spin

thread = threading.Thread(target=worker, daemon=True)
thread.start()

win = "picking"
cv2.namedWindow(win, cv2.WINDOW_NORMAL)
cv2.resizeWindow(win, 1348, 1011)
detach = False
try:
    while thread.is_alive():
        ok, frame = over_cam.read()
        if ok:
            with lock:
                s = dict(snap)
            lines = [f"state: {s['state'].value}",
                     f"target: {s['target']}",
                     "PAUSED" if pause.is_set() else "running"]
            vis = overlays.annotate(
                frame, cuboid_df=s["df"], pickable=s["pickable"],
                isolated=s["isolated"], floater_zones=s["zones"],
                floater_radius=cfg.floater_zone_radius_px,
                circle_center=cfg.circle_center, circle_radius=cfg.circle_radius,
                status_lines=lines)
            cv2.imshow(win, vis)

        key = cv2.waitKey(20) & 0xFF
        if key == ord("p"):
            pause.clear() if pause.is_set() else pause.set()
        elif key == 27:                     # Esc: stop the routine
            stop.set()
        elif key == ord("q"):               # leave the window, keep running
            detach = True
            break
        elif key == ord("r"):               # resume after NEEDS_OPERATOR
            session.resume()
finally:
    cv2.destroyWindow(win)
    if not detach:
        stop.set()
        thread.join(timeout=5)
        session.close()                     # detach the clip recorder, if any
        openapi.retract_axis("leftZ")
    print("state:", session.state.value, "(detached, still running)" if detach else "")

## 7. Shutting down

In [ ]:
openapi.retract_axis("leftZ")
cams.close_all()